In [1]:
import pandas as pd
import numpy as np

print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)
print("Environment ready!")

Pandas version: 2.2.2
NumPy version: 1.26.4
Environment ready!


In [3]:
# Reproducibility
np.random.seed(42)

# 1. Production Lines
production_lines = pd.DataFrame({
    "Line_ID": [101, 102, 103, 104, 105],
    "Line_Name": [
        "Assembly Line A",
        "Assembly Line B",
        "Packaging Line",
        "Machining Line",
        "Testing Line"
    ],
    "Location": [
        "Plant A", "Plant A", "Plant B",
        "Plant B", "Plant C"
    ],
    "Capacity_Per_Shift": [500, 450, 600, 400, 350]
})

# 2. Machines
machine_types = [
    "Assembly Machine",
    "CNC Machine",
    "Packaging Machine",
    "Testing Machine",
    "Conveyor System"
]

machines = pd.DataFrame({
    "Machine_ID": range(1001, 1021),
    "Machine_Name": [
        f"Machine_{i}" for i in range(1, 21)
    ],
    "Line_ID": np.repeat(
        [101, 102, 103, 104, 105], 4
    ),
    "Machine_Type": np.random.choice(
        machine_types, 20
    ),
    "Installation_Date": pd.to_datetime(
        np.random.choice(
            pd.date_range("2020-01-01", "2025-01-01"),
            20
        )
    ).date
})

# 3. Products
products = pd.DataFrame({
    "Product_ID": range(2001, 2016),
    "Product_Name": [
        f"Industrial_Component_{i}" for i in range(1, 16)
    ],
    "Product_Category": np.random.choice(
        ["Automotive", "Electrical", "Mechanical"],
        15
    ),
    "Standard_Cycle_Time_Seconds": np.random.uniform(
        20, 120, 15
    ).round(2)
})

# Preview datasets
print("Production Lines:", production_lines.shape)
print("Machines:", machines.shape)
print("Products:", products.shape)

display(production_lines.head())
display(machines.head())
display(products.head())


Production Lines: (5, 4)
Machines: (20, 5)
Products: (15, 4)


,Line_ID,Line_Name,Location,Capacity_Per_Shift
0,101,Assembly Line A,Plant A,500
1,102,Assembly Line B,Plant A,450
2,103,Packaging Line,Plant B,600
3,104,Machining Line,Plant B,400
4,105,Testing Line,Plant C,350


,Machine_ID,Machine_Name,Line_ID,Machine_Type,Installation_Date
0,1001,Machine_1,101,Testing Machine,2023-08-30
1,1002,Machine_2,101,Conveyor System,2020-01-22
2,1003,Machine_3,101,Packaging Machine,2020-09-09
3,1004,Machine_4,101,Conveyor System,2022-01-17
4,1005,Machine_5,102,Conveyor System,2022-05-06


,Product_ID,Product_Name,Product_Category,Standard_Cycle_Time_Seconds
0,2001,Industrial_Component_1,Automotive,29.77
1,2002,Industrial_Component_2,Mechanical,88.42
2,2003,Industrial_Component_3,Automotive,64.02
3,2004,Industrial_Component_4,Mechanical,32.20
4,2005,Industrial_Component_5,Mechanical,69.52


In [5]:

import os

os.makedirs("../Dataset", exist_ok=True)

production_lines.to_csv(
    "../Dataset/Production_Lines.csv", index=False
)

machines.to_csv(
    "../Dataset/Machines.csv", index=False
)

products.to_csv(
    "../Dataset/Products.csv", index=False
)

print("Master datasets exported successfully!")


Master datasets exported successfully!


In [7]:

# 4. Production Orders

num_orders = 1000

production_orders = pd.DataFrame({
    "Order_ID": range(5001, 5001 + num_orders),
    "Product_ID": np.random.choice(
        products["Product_ID"], num_orders
    ),
    "Line_ID": np.random.choice(
        production_lines["Line_ID"], num_orders
    ),
    "Order_Date": pd.to_datetime(
        np.random.choice(
            pd.date_range("2025-01-01", "2025-12-31"),
            num_orders
        )
    ).date,
    "Planned_Quantity": np.random.randint(
        400, 1001, num_orders
    )
})

display(production_orders.head())
print("Production Orders:", production_orders.shape)


,Order_ID,Product_ID,Line_ID,Order_Date,Planned_Quantity
0,5001,2002,104,2025-06-13,563
1,5002,2010,101,2025-10-28,884
2,5003,2014,103,2025-10-30,996
3,5004,2004,103,2025-10-14,471
4,5005,2014,103,2025-01-10,966


Production Orders: (1000, 5)


In [9]:

# 5. Production Records

production_records_list = []

shifts = ["Morning", "Evening", "Night"]
record_id = 1

for _, order in production_orders.iterrows():

    line_machines = machines[
        machines["Line_ID"] == order["Line_ID"]
    ]

    quantities = np.random.multinomial(
        order["Planned_Quantity"],
        [0.20, 0.20, 0.20, 0.20, 0.20]
    )

    for i in range(5):

        total_units = int(quantities[i])

        defect_rate = np.random.uniform(0.01, 0.10)

        rejected_units = int(
            round(total_units * defect_rate)
        )

        good_units = total_units - rejected_units

        machine_id = int(
            np.random.choice(line_machines["Machine_ID"])
        )

        production_records_list.append({
            "Record_ID": record_id,
            "Order_ID": order["Order_ID"],
            "Machine_ID": machine_id,
            "Shift_Name": shifts[i % 3],
            "Production_Date": order["Order_Date"],
            "Total_Units": total_units,
            "Good_Units": good_units,
            "Rejected_Units": rejected_units
        })

        record_id += 1

production_records = pd.DataFrame(
    production_records_list
)

display(production_records.head())
print("Production Records:", production_records.shape)


,Record_ID,Order_ID,Machine_ID,Shift_Name,Production_Date,Total_Units,Good_Units,Rejected_Units
0,1,5001,1014,Morning,2025-06-13,124,121,3
1,2,5001,1014,Evening,2025-06-13,116,107,9
2,3,5001,1015,Night,2025-06-13,104,95,9
3,4,5001,1015,Morning,2025-06-13,99,96,3
4,5,5001,1014,Evening,2025-06-13,120,113,7


Production Records: (5000, 8)


In [11]:

# 6. Quality Inspections

defect_types = [
    "Surface Defect",
    "Dimensional Error",
    "Assembly Failure",
    "Material Defect",
    "Electrical Failure"
]

quality_inspections = pd.DataFrame({
    "Inspection_ID": range(1, 5001),
    "Order_ID": production_records["Order_ID"].values,
    "Inspection_Date": production_records[
        "Production_Date"
    ].values,
    "Inspected_Quantity": production_records[
        "Total_Units"
    ].values,
    "Defect_Quantity": production_records[
        "Rejected_Units"
    ].values,
    "Defect_Type": [
        np.random.choice(defect_types) if rejected > 0
        else "No Defect"
        for rejected in production_records["Rejected_Units"]
    ]
})

display(quality_inspections.head())
print("Quality Inspections:", quality_inspections.shape)


,Inspection_ID,Order_ID,Inspection_Date,Inspected_Quantity,Defect_Quantity,Defect_Type
0,1,5001,2025-06-13,124,3,Dimensional Error
1,2,5001,2025-06-13,116,9,Surface Defect
2,3,5001,2025-06-13,104,9,Dimensional Error
3,4,5001,2025-06-13,99,3,Assembly Failure
4,5,5001,2025-06-13,120,7,Assembly Failure


Quality Inspections: (5000, 6)


In [13]:

# 7. Downtime Logs

downtime_reasons = [
    "Machine Breakdown",
    "Preventive Maintenance",
    "Power Failure",
    "Material Shortage",
    "Tool Change",
    "Setup Adjustment"
]

downtime_logs = pd.DataFrame({
    "Downtime_ID": range(1, 1001),
    "Machine_ID": np.random.choice(
        machines["Machine_ID"], 1000
    ),
    "Downtime_Date": pd.to_datetime(
        np.random.choice(
            pd.date_range("2025-01-01", "2025-12-31"),
            1000
        )
    ).date,
    "Downtime_Minutes": np.random.randint(
        5, 181, 1000
    ),
    "Reason": np.random.choice(
        downtime_reasons, 1000
    )
})

display(downtime_logs.head())
print("Downtime Logs:", downtime_logs.shape)


,Downtime_ID,Machine_ID,Downtime_Date,Downtime_Minutes,Reason
0,1,1014,2025-12-22,33,Tool Change
1,2,1011,2025-10-06,37,Material Shortage
2,3,1018,2025-09-11,32,Tool Change
3,4,1005,2025-05-22,99,Preventive Maintenance
4,5,1015,2025-06-14,87,Tool Change


Downtime Logs: (1000, 5)


In [15]:

# Export transaction datasets

production_orders.to_csv(
    "../Dataset/Production_Orders.csv", index=False
)

production_records.to_csv(
    "../Dataset/Production_Records.csv", index=False
)

quality_inspections.to_csv(
    "../Dataset/Quality_Inspections.csv", index=False
)

downtime_logs.to_csv(
    "../Dataset/Downtime_Logs.csv", index=False
)

print("All transaction datasets exported successfully!")


All transaction datasets exported successfully!


In [17]:
datasets = {
    "Production_Lines": production_lines,
    "Machines": machines,
    "Products": products,
    "Production_Orders": production_orders,
    "Production_Records": production_records,
    "Quality_Inspections": quality_inspections,
    "Downtime_Logs": downtime_logs
}

for name, df in datasets.items():
    print(f"{name}: {len(df)} records")

Production_Lines: 5 records
Machines: 20 records
Products: 15 records
Production_Orders: 1000 records
Production_Records: 5000 records
Quality_Inspections: 5000 records
Downtime_Logs: 1000 records


In [19]:

# 1. Check missing values
print("MISSING VALUE CHECK")
for name, df in datasets.items():
    print(name, ":", df.isnull().sum().sum())

# 2. Check duplicate primary keys
print("\nDUPLICATE ID CHECK")
id_columns = {
    "Production_Lines": "Line_ID",
    "Machines": "Machine_ID",
    "Products": "Product_ID",
    "Production_Orders": "Order_ID",
    "Production_Records": "Record_ID",
    "Quality_Inspections": "Inspection_ID",
    "Downtime_Logs": "Downtime_ID"
}

for name, column in id_columns.items():
    print(name, ":", datasets[name][column].duplicated().sum())

# 3. Check production quantity consistency
print("\nPRODUCTION QUANTITY CHECK")

print(
    "Quantity mismatches:",
    (
        production_records["Total_Units"]
        != production_records["Good_Units"]
        + production_records["Rejected_Units"]
    ).sum()
)

# 4. Check machine-line relationships
print("\nMACHINE-LINE CHECK")

record_lines = production_records.merge(
    production_orders[["Order_ID", "Line_ID"]],
    on="Order_ID"
).merge(
    machines[["Machine_ID", "Line_ID"]],
    on="Machine_ID",
    suffixes=("_order", "_machine")
)

print(
    "Machine-line mismatches:",
    (record_lines["Line_ID_order"]
     != record_lines["Line_ID_machine"]).sum()
)


MISSING VALUE CHECK
Production_Lines : 0
Machines : 0
Products : 0
Production_Orders : 0
Production_Records : 0
Quality_Inspections : 0
Downtime_Logs : 0

DUPLICATE ID CHECK
Production_Lines : 0
Machines : 0
Products : 0
Production_Orders : 0
Production_Records : 0
Quality_Inspections : 0
Downtime_Logs : 0

PRODUCTION QUANTITY CHECK
Quantity mismatches: 0

MACHINE-LINE CHECK
Machine-line mismatches: 0
